In [ ]:
import pandas as pd
import json
import time
import requests
import os


API_KEY = "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"

URL = "https://openrouter.ai/api/v1/chat/completions"

HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

Models evaluating:
- DeepSeek — DeepSeek V3
- Anthropic — Claude 3.5 Sonnet                              switch Claude 3.5 Sonnet with Claude Sonnet 4 
- Google — Gemini 2.5 Flash       
- Meta — Llama 3.1 70B Instruct
- Mistral - Mistral 

In [ ]:
eval_df = pd.read_csv("datasets/yoruba_proverbs_eval_final.csv", encoding="utf-8-sig")
print(eval_df.shape)

In [ ]:
MODEL_KEYS = ["deepseek", "claude", "gemini", "mistral", "llama"]

OPENROUTER_MODELS = {
    "deepseek": "deepseek/deepseek-chat-v3-0324",
    "claude":   "anthropic/claude-sonnet-4",
    "gemini":   "google/gemini-2.5-flash",
    "llama":    "meta-llama/llama-3.1-70b-instruct",
    "mistral":  "mistralai/mistral-large-2512",  # cost-efficient “Large”
}

for m in MODEL_KEYS:
    eval_df[f"{m}_translation"] = ""
    eval_df[f"{m}_interpretation"] = ""
    eval_df[f"{m}_raw"] = ""     # store raw model output (debug)
    eval_df[f"{m}_error"] = ""   # store errors (rate limit, json issues, etc.)
    eval_df[f"{m}_prompt_tokens"] = ""
    eval_df[f"{m}_completion_tokens"] = ""
    eval_df[f"{m}_total_tokens"] = ""

eval_df.head(2)
eval_df.columns.tolist()

In [ ]:
PROMPT = """You are a Yoruba language and culture expert.

Given this Yoruba proverb, return VALID JSON ONLY with exactly these keys:
{{
  "translation": "literal English translation (surface meaning)",
  "interpretation": "one concise sentence explaining the meaning or lesson"
}}

Rules:
- Output JSON only. No markdown. No extra keys.
- Do not invent details not supported by the proverb.
- Keep interpretation concise and culturally grounded.
- Translation must be literal (surface meaning).
- Interpretation must explain the meaning in one concise sentence.

Proverb: "{proverb}"
"""

In [ ]:
import json
import re

def parse_json_strict(text: str) -> dict:
    """
    Extract the first valid JSON object from model output.
    Ensures required keys exist.
    """

    # Find first JSON-looking block
    match = re.search(r"\{.*?\}", text, flags=re.DOTALL)
    if not match:
        raise ValueError("No JSON object found in model output.")

    json_str = match.group(0)

    try:
        obj = json.loads(json_str)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON format: {e}")

    # Validate required keys
    if not isinstance(obj, dict):
        raise ValueError("Parsed JSON is not an object.")

    if "translation" not in obj or "interpretation" not in obj:
        raise ValueError("JSON missing required keys.")

    return obj

In [ ]:
sample = eval_df.head(10).copy()
sample["yoruba"].tolist()[:3]

In [ ]:
def call_model(model_key, proverb_text):

    model_id = OPENROUTER_MODELS[model_key]

    user_prompt = PROMPT.format(proverb=proverb_text)

    payload = {
        "model": model_id,
        "messages": [
            {"role": "user", "content": user_prompt}
        ],
        "temperature": 0.2,
        "max_tokens": 250   
    }

    session = requests.Session()
    session.trust_env = False  # keeps 401 from coming back

    r = session.post(URL, headers=HEADERS, json=payload, timeout=90)

    if r.status_code != 200:
        raise RuntimeError(f"{r.status_code}: {r.text}")

    j = r.json()
    raw = j["choices"][0]["message"]["content"]
    usage = j.get("usage", {})

    parsed = parse_json_strict(raw)

    return {
        "translation": parsed["translation"],
        "interpretation": parsed["interpretation"],
        "raw": raw,
        "usage": usage
    }

In [ ]:
run_df = eval_df.copy()

# Make sure your output columns exist on run_df
for m in MODEL_KEYS:
    for suffix in ["translation","interpretation","raw","error","prompt_tokens","completion_tokens","total_tokens"]:
        col = f"{m}_{suffix}"
        if col not in run_df.columns:
            run_df[col] = ""

In [ ]:
CHECKPOINT_EVERY = 100

# Ensure these columns are empty strings (not NaN) in run_df
for i, row in enumerate(run_df.itertuples()):
    proverb = getattr(row, "yoruba")
    idx = row.Index

    for m in MODEL_KEYS:
        # check run_df, not eval_df
        existing = str(run_df.at[idx, f"{m}_translation"]).strip()
        if existing != "" and existing.lower() != "nan":
            continue

        try:
            print(f"[{i+1}/{len(run_df)}] Calling {m} for idx={idx}")
            out = call_model(m, proverb)

            # write to run_df, not eval_df
            run_df.at[idx, f"{m}_translation"] = out["translation"]
            run_df.at[idx, f"{m}_interpretation"] = out["interpretation"]
            run_df.at[idx, f"{m}_raw"] = out["raw"]

            usage = out.get("usage", {})
            run_df.at[idx, f"{m}_prompt_tokens"] = usage.get("prompt_tokens", "")
            run_df.at[idx, f"{m}_completion_tokens"] = usage.get("completion_tokens", "")
            run_df.at[idx, f"{m}_total_tokens"] = usage.get("total_tokens", "")

        except Exception as e:
            run_df.at[idx, f"{m}_error"] = str(e)

    # SMALL DELAY AFTER EACH PROVERB
    time.sleep(0.2)

    if (i + 1) % CHECKPOINT_EVERY == 0:
        run_df.to_csv("full_dataset_checkpoint.csv", index=False, encoding="utf-8-sig")
        print("Full dataset checkpoint saved:", i + 1)
        time.sleep(0.5)

run_df.to_csv("full_dataset_results.csv", index=False, encoding="utf-8-sig")
run_df.to_parquet("full_dataset_results.parquet", index=False)
print("Saved full_dataset_results.csv and full_dataset_results.parquet")

In [ ]:
mask = eval_df["deepseek_translation"].astype(str).str.strip() != ""
eval_df[mask].index.max()

In [ ]:
run_df[[f"{m}_translation" for m in MODEL_KEYS]].head(10)